# Linearity of steering deviations

Test whether linear predictors describe the effect of fixed A-LQR pushes as their size changes. Every measurement is a single forward pass on one unpadded prompt, at generation step 0; there is no generation. Rollout measures accumulated prediction error; one-step uses the true incoming deviation to isolate each block. The final state is measured before the final normalization.

| Symbol | Meaning |
|---|---|
| $l, L$ | Block index $l=0,\ldots,L-1$ and number of blocks. State index $L$ denotes the input to the final norm. States without a token-position argument refer to the prompt's last position. |
| $H, I, y$ | Hidden size, identity matrix, and an arbitrary hidden-state vector. |
| $x_l^{\mathrm{nom}}$ | State entering block $l$ without steering. |
| $u_l$ | Reference push added to block $l$'s output, zero at unsteered blocks. |
| $x_l$ | State entering block $l$ with pushes. |
| $d_l=x_l-x_l^{\mathrm{nom}}$ | True deviation. |
| $\hat d_l$ | Predicted deviation. |
| $E_l, v_l=E_l/\|E_l\|$ | Cached concept vector and unit concept direction; the cache also supplies the final-state direction. |
| $\Pi_l=I-v_lv_l^\top$ | Projection removing the concept component. |
| $P_l$ | Predictor's linear model of block $l$. |
| $s$ | Scale multiplying the fixed reference pushes. |
| $m$ (`every`) | Actuation interval: steer every $m$-th block. |
| $\lambda$ | A-LQR target strength, fixed at 4 when recording reference pushes. |
| $Q,R,Q_F$ | State, control, and terminal costs; configured scalar weights multiply identity matrices. |
| $\|\cdot\|$, $\top$ | Euclidean norm and transpose. |
| $d_L(s)$ | True final-state deviation when the reference pushes are multiplied by $s$. |
| $t, x_l^{\mathrm{nom}}(t)$ | Token position and unsteered state entering block $l$ at that position. |
| $X_l,Y_l,N$ | Data matrices with shape $H \times N$; columns are training samples. |
| $m_x,m_y,X_c,Y_c$ | Training-column means and centered matrices $X_c=X_l-m_x$, $Y_c=Y_l-m_y$. |
| $r,U_r,S_r,V_r$ | Truncation rank and retained SVD factors: $X_c \approx U_r S_r V_r^\top$ for vanilla DMD, or the same factorization of uncentered $X_l$ for steering DMD. |
| $\Delta_l,L_l$ | Learned correction $\Delta_l=L_l U_r^\top$ and its left factor. The fitted predictor is $P_l=I+\Delta_l$; each factor has shape $H \times r$, capped at that block's numerical data rank. |

Rollout: $\hat d_0=0$, $\hat d_{l+1}=P_l\hat d_l+s u_l$.
One-step: $\hat d_{l+1}=P_l d_l+s u_l$.

Metrics are $\|d_l\|/\|x_l^{\mathrm{nom}}\|$ (`rel_dev`),
$\|\hat d_l-d_l\|/\|d_l\|$ (`err_full`),
$\|\Pi_l(\hat d_l-d_l)\|/\|\Pi_l d_l\|$ (`err_perp`),
$\|\Pi_l\hat d_l\|/\|\Pi_l d_l\|$ (`ratio_perp`), and the cosine between
$\Pi_l\hat d_l$ and $\Pi_l d_l$ (`cos_perp`). Zero true-deviation states are omitted; undefined perpendicular ratios/cosines remain NaN.

In [ ]:
from pathlib import Path
from contextlib import contextmanager
from functools import partial
import csv, random, sys
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.nn.attention import SDPBackend, sdpa_kernel
from tqdm.auto import tqdm

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'llm' / 'steering.py').is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from llm.model import load_model, blocks, LayerInputs
from llm.concepts import load_concept_prompts, SENTENCES_CSV
from llm.jacobian import _LAYER_KWARGS
from llm.steering import LQRSteering

MODEL_NAME = 'Qwen/Qwen3-4B'
CONCEPT = 'football'
SEED = 0
DIRECTION_SAMPLES = 200
JACOBIAN_SAMPLES = 50
MAX_CTX = 32
RANDOM_SEED = 0
Q, R, QF = 0.1, 10, 0.1
NUM_PROMPTS = 100
EVERY = (1, 2)
LAMBDA = 4
SCALES = (0, 0.01, 0.03, 0.1, 0.3, 1, 1.5)
PREDICTORS = ('identity', 'mean_jacobian', 'prompt_jacobian', 'random',
              'vanilla_dmd', 'steering_dmd', 'steering_dmd_s1')
TRAINING_PROMPTS = 1000
TRAINING_SEED = SEED + 2
DMD_SCALES = (0.1, 0.3, 1.0)
DMD_RANKS = (8, 32, 128, 512, 2048)
DMD_NAMES = ('vanilla_dmd', 'steering_dmd', 'steering_dmd_s1')

ARTIFACTS = REPO_ROOT / 'artifacts' / MODEL_NAME.rsplit('/', 1)[-1].lower()
RUN_DIR = ARTIFACTS / 'linearity_test' / f'q{Q:g}_r{R:g}_qf{QF:g}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Results:', RUN_DIR)


## Model, caches, and prompts

Load float32 explicitly, disable TF32, and use math attention for both forwards and JVPs. No BF16 fallback is performed silently. Scalar metric reductions use float64. E and the Mean Jacobian are read-only caches from notebook 03; the cached Mean Jacobian is deliberately not recomputed for these prompts.

Reconstruct the direction's seeded background sample, then choose 100 of those prompts. Exclude the Jacobian sample and duplicate inputs after 32-token truncation. Process each prompt separately with no padding.

In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_float32_matmul_precision('highest')
print('Free GPU GiB:', round(torch.cuda.mem_get_info()[0] / 2**30, 2))
tokenizer, model = load_model(MODEL_NAME, dtype=torch.float32)
model.requires_grad_(False)
layers = list(blocks(model))
L = len(layers)
H = model.get_input_embeddings().weight.shape[1]
device = model.device
assert next(model.parameters()).dtype == torch.float32
print(f'{MODEL_NAME}: {L} blocks, hidden {H}, {next(model.parameters()).dtype}, {device}')

E_PATH = ARTIFACTS / f'E_{CONCEPT}_n{DIRECTION_SAMPLES}_seed{SEED}.pt'
A_PATH = ARTIFACTS / f'A_{CONCEPT}_n{JACOBIAN_SAMPLES}_ctx{MAX_CTX}_seed{SEED}.pt'
E = torch.load(E_PATH, map_location='cpu', weights_only=True)
A_jac = torch.load(A_PATH, map_location='cpu', weights_only=True)
assert E.shape == (L + 1, H) and A_jac.shape == (L, H, H)
B_random = torch.randn(L, H, H, generator=torch.Generator().manual_seed(RANDOM_SEED))
size = torch.linalg.matrix_norm(A_jac - torch.eye(H)) / torch.linalg.matrix_norm(B_random)
A_random = torch.eye(H) + size[:, None, None] * B_random
del B_random
A_jac, A_random = A_jac.to(device), A_random.to(device)
A_identity = torch.eye(H, device=device).expand(L, -1, -1)
v = E.to(device=device, dtype=torch.float64)
v = v / v.norm(dim=-1, keepdim=True).clamp_min(1e-12)
controllers = {every: LQRSteering(model, A_identity, E, Q, R, QF, every=every) for every in EVERY}

random.seed(SEED)
_, background = load_concept_prompts(CONCEPT, DIRECTION_SAMPLES)
random.seed(SEED + 1)
jacobian_prompts, _ = load_concept_prompts(CONCEPT, JACOBIAN_SAMPLES)
jacobian_ids = {tuple(tokenizer(text, truncation=True, max_length=MAX_CTX)['input_ids'])
                for text in jacobian_prompts}
candidates = {}
for text in background:
    tokens = tuple(tokenizer(text, truncation=True, max_length=MAX_CTX)['input_ids'])
    if tokens not in jacobian_ids:
        candidates.setdefault(tokens, text)
prompts = random.Random(SEED).sample(list(candidates.items()), NUM_PROMPTS)
print(f'{len(prompts)} distinct background inputs; max {max(len(tokens) for tokens, _ in prompts)} tokens; no Jacobian overlap.')
print('E:', E_PATH.name, '| Mean Jacobian:', A_PATH.name)


## Forward capture and last-token JVP

The block callable in `llm/jacobian.py` is nested inside the Jacobian routines, not importable. The small closure below uses that same concatenate-prefix/call-block/select-last pattern and its existing kwargs whitelist. Prefix positions stay at their nominal values. Vectorizing tangent directions does not batch different prompts.

The push recorder wraps the existing controller's `push` method only to copy the returned vector in the output dtype. The original controller still performs the reference additions. Replays use fixed vectors, never feedback recomputation.

In [ ]:
def forward_states(inputs, pushes=None, capture_kwargs=False):
    layer_kwargs = [{} for _ in layers]
    handles = []

    def capture(layer, module, args, kwargs):
        layer_kwargs[layer] = {key: kwargs[key] for key in _LAYER_KWARGS if kwargs.get(key) is not None}

    def add_push(layer, module, args, kwargs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        hidden = hidden.clone()
        hidden[:, -1] += pushes[layer].to(hidden.dtype)
        return (hidden, *output[1:]) if isinstance(output, tuple) else hidden

    try:
        for layer, block in enumerate(layers):
            if capture_kwargs:
                handles.append(block.register_forward_pre_hook(partial(capture, layer), with_kwargs=True))
            if pushes is not None:
                handles.append(block.register_forward_hook(partial(add_push, layer), with_kwargs=True))
        with LayerInputs(model) as record:
            model.base_model(**inputs, use_cache=False)
        return record.states, layer_kwargs
    finally:
        for handle in handles:
            handle.remove()


@contextmanager
def record_pushes(controller):
    original_push = controller.push
    pushes = torch.zeros(L, H, dtype=torch.float32, device=device)

    def save_push(layer, x):
        u = original_push(layer, x)
        pushes[layer].copy_(u[0, -1].to(x.dtype))
        return u

    controller.push = save_push
    try:
        yield pushes
    finally:
        del controller.push


def last_token_block(block, x, kwargs):
    def last_output(last):
        output = block(torch.cat((x[:-1], last[None]))[None], **kwargs)
        return (output[0] if isinstance(output, tuple) else output)[0, -1]
    return last_output


def predict_deviations(clean_states, layer_kwargs, pushes, deviations, scales):
    rollout = {name: torch.zeros(L + 1, H, device=device) for name in PREDICTORS}
    one_step = {name: torch.zeros_like(deviations) for name in PREDICTORS}
    for layer, block in enumerate(layers):
        x = clean_states[layer][0]
        last_output = last_token_block(block, x, layer_kwargs[layer])
        applied = scales[:, None] * pushes[layer]
        for name in PREDICTORS:
            directions = torch.cat((rollout[name][layer:layer + 1], deviations[:, layer]))
            if name == 'identity':
                propagated = directions
            elif name == 'mean_jacobian':
                propagated = directions @ A_jac[layer].T
            elif name == 'random':
                propagated = directions @ A_random[layer].T
            elif name == 'prompt_jacobian':
                propagated = torch.vmap(lambda y: torch.func.jvp(last_output, (x[-1],), (y,))[1])(directions)
            else:
                propagated = apply_dmd(directions, dmd_factors[name][layer])
            rollout[name][layer + 1] = propagated[0] + pushes[layer]
            one_step[name][:, layer + 1] = propagated[1:] + applied
    return {name: {'rollout': scales[:, None, None] * rollout[name][None], 'one_step': one_step[name]}
            for name in PREDICTORS}


## DMD training pool and cache policy

Keep the original 100 test prompts unchanged. Draw a separate pool from the full non-concept CSV, deduplicating and excluding test/Jacobian inputs after tokenization and truncation. Hold out 10% by prompt; all positions, actuation intervals, and scales from a prompt stay in the same split.

Existing predictor caches are loaded, never refitted. A configuration mismatch stops rather than overwriting a cache. Cached fits retain the exact prompt split, selected rank, validation curve, effective ranks, and factors.


In [ ]:
test_ids = {tokens for tokens, _ in prompts}
with SENTENCES_CSV.open(newline='', encoding='utf-8') as stream:
    training_texts = [row['sentence'] for row in csv.DictReader(stream)
                      if row['concept'] != f'{CONCEPT}.NOUN']
training_candidates = {}
for text in training_texts:
    tokens = tuple(tokenizer(text, truncation=True, max_length=MAX_CTX)['input_ids'])
    if tokens not in test_ids and tokens not in jacobian_ids:
        training_candidates.setdefault(tokens, text)
training_pool = random.Random(TRAINING_SEED).sample(
    list(training_candidates.items()), min(TRAINING_PROMPTS, len(training_candidates)))
validation_count = max(1, int(np.ceil(0.1 * len(training_pool))))
validation_prompts = training_pool[:validation_count]
fit_prompts = training_pool[validation_count:]
assert fit_prompts and validation_prompts
assert {tokens for tokens, _ in fit_prompts}.isdisjoint(tokens for tokens, _ in validation_prompts)
assert set(training_candidates).isdisjoint(test_ids | jacobian_ids)

DMD_DIR = RUN_DIR / 'dmd'
DMD_DIR.mkdir(parents=True, exist_ok=True)
cache_paths = {name: DMD_DIR / f'{name}_{CONCEPT}.pt' for name in DMD_NAMES}
fit_config = {
    'model': MODEL_NAME, 'concept': CONCEPT, 'dtype': 'float32', 'attention': 'math',
    'max_ctx': MAX_CTX, 'q': Q, 'r': R, 'qf': QF, 'lambda': LAMBDA,
    'every': list(EVERY), 'scales': list(DMD_SCALES), 'ranks': list(DMD_RANKS),
    'seed': TRAINING_SEED,
    'direction_cache': [str(E_PATH), E_PATH.stat().st_size, E_PATH.stat().st_mtime_ns],
    'fit_prompts': fit_prompts, 'validation_prompts': validation_prompts,
    'test_prompts': prompts, 'jacobian_inputs': sorted(jacobian_ids),
}
print(f'DMD pool: {len(training_pool)} / {TRAINING_PROMPTS} requested; '
      f'{len(fit_prompts)} fitting, {len(validation_prompts)} validation, {len(prompts)} unchanged test prompts.')
print(f'Eligible non-concept inputs: {len(training_candidates)}')


## Collect fitting and validation responses

Vanilla DMD uses unsteered positions $t \geq 1$ from fitting prompts only. Steering DMD uses last-position deviations from both actuation intervals: columns of $X_l$ are $d_l$, and columns of $Y_l$ are $d_{l+1}-s u_l$. The scale-one variant selects only those fitting responses; neither steering fit is centered. All three fits are validated on the same held-out prompts, both actuation intervals, and scales 0.1, 0.3, and 1.

Temporary memory-mapped arrays keep all-position training data out of RAM. They are removed after fitting; only the fitted predictors and their selection reports are cached.


In [ ]:
def collect_dmd_data(directory, needed):
    directory = Path(directory)
    collect_vanilla = 'vanilla_dmd' in needed
    collect_steering = any(name != 'vanilla_dmd' for name in needed)
    train_scales = DMD_SCALES if 'steering_dmd' in needed else (1.0,)
    train_cases = len(fit_prompts) * len(EVERY)
    train_samples = train_cases * len(train_scales)
    validation_samples = len(validation_prompts) * len(EVERY) * len(DMD_SCALES)
    data = {
        'scales': np.tile(train_scales, train_cases),
        'cases': np.repeat(np.arange(train_cases), len(train_scales)),
        'validation_d': torch.empty(L + 1, validation_samples, H),
        'validation_u': torch.empty(L, validation_samples, H),
    }
    if collect_vanilla:
        token_count = sum(len(tokens) - 1 for tokens, _ in fit_prompts)
        data['nominal'] = np.lib.format.open_memmap(
            directory / 'nominal.npy', mode='w+', dtype=np.float32, shape=(L + 1, token_count, H))
    if collect_steering:
        data['deviations'] = np.lib.format.open_memmap(
            directory / 'deviations.npy', mode='w+', dtype=np.float32, shape=(L + 1, train_samples, H))
        data['pushes'] = np.lib.format.open_memmap(
            directory / 'pushes.npy', mode='w+', dtype=np.float32, shape=(L, train_cases, H))
    token_offset = 0
    with torch.no_grad(), sdpa_kernel(SDPBackend.MATH), tqdm(
            total=len(fit_prompts) + len(validation_prompts), desc='DMD training/validation states', unit='prompt') as progress:
        for split, split_prompts in (('fit', fit_prompts), ('validation', validation_prompts)):
            for prompt_id, (tokens, _) in enumerate(split_prompts):
                input_ids = torch.tensor([tokens], device=device)
                inputs = {'input_ids': input_ids, 'attention_mask': torch.ones_like(input_ids)}
                states, _ = forward_states(inputs)
                nominal = torch.stack([state[0, -1] for state in states])
                if split == 'fit' and collect_vanilla:
                    count = len(tokens) - 1
                    data['nominal'][:, token_offset:token_offset + count] = torch.stack(
                        [state[0, 1:] for state in states]).cpu().numpy()
                    token_offset += count
                del states
                if split == 'validation' or collect_steering:
                    selected_scales = DMD_SCALES if split == 'validation' else train_scales
                    for every_index, every in enumerate(EVERY):
                        case = prompt_id * len(EVERY) + every_index
                        controller = controllers[every]
                        with record_pushes(controller) as pushes, controller.active(LAMBDA):
                            forward_states(inputs)
                        if split == 'fit':
                            data['pushes'][:, case] = pushes.cpu().numpy()
                        for scale_index, scale in enumerate(selected_scales):
                            sample_index = case * len(selected_scales) + scale_index
                            states, _ = forward_states(inputs, pushes=scale * pushes)
                            deviation = torch.stack([state[0, -1] for state in states]) - nominal
                            if split == 'fit':
                                data['deviations'][:, sample_index] = deviation.cpu().numpy()
                            else:
                                data['validation_d'][:, sample_index].copy_(deviation.cpu())
                                data['validation_u'][:, sample_index].copy_((scale * pushes).cpu())
                            del states
                progress.set_postfix(split=split)
                progress.update(1)
    return data


## Change-form SVD and rank selection

For vanilla DMD, center both matrices using fitting-data means and set
$$L_l=(Y_c-X_c)V_r S_r^{-1}.$$
For steering DMD, use
$$L_l=(Y_l-X_l)V_r S_r^{-1}.$$
Apply the predictor as $P_l y=y+L_l(U_r^\top y)$; no fitted bias is applied to deviations.

Each candidate rank is capped by the block's numerical data rank, using the standard float32 SVD tolerance: largest singular value times the larger matrix dimension times machine epsilon. Empty steering inputs at block 0 produce a rank-zero correction, hence identity. Effective ranks count the singular values containing 90% and 99% of the total squared spectrum, after centering for vanilla DMD.

For each candidate, compute one-step off-concept errors using the held-out true incoming deviations and applied pushes. Take a median over held-out prompt/interval/scale responses separately at each block, then average those block medians. Select one requested rank per fit; per-block caps still apply. Ties choose the smaller rank. The fitted factors use fitting prompts only, not a refit on validation prompts.

Fit reports index operator blocks from 0 to $L-1$; a block's validation error is measured at its output state $l+1$. The test metrics and depth plots index states from 1 to $L$, including the final-norm input.


In [ ]:
def apply_dmd(directions, factors, rank=None):
    left, basis = factors
    if rank is not None:
        left, basis = left[:, :rank], basis[:, :rank]
    if basis.shape[1] == 0:
        return directions
    return directions + (directions @ basis) @ left.T


def change_form_factors(X, Y, centered):
    means = None
    if centered:
        mx, my = X.mean(dim=1, keepdim=True), Y.mean(dim=1, keepdim=True)
        means = (mx.cpu(), my.cpu())
        X, Y = X - mx, Y - my
    if X.shape[1] == 0:
        empty = X.new_empty(X.shape[0], 0)
        return (empty, empty), means, {'samples': 0, 'data_rank': 0, 'rank_90': 0, 'rank_99': 0}
    U, singular, Vh = torch.linalg.svd(X, full_matrices=False, driver='gesvd' if X.is_cuda else None)
    tolerance = singular[0] * max(X.shape) * torch.finfo(X.dtype).eps
    data_rank = int((singular > tolerance).sum().item())
    retained = min(max(DMD_RANKS), data_rank)
    left = ((Y - X) @ Vh[:retained].T) / singular[:retained]
    basis = U[:, :retained]
    energy = singular.double().square().cumsum(dim=0)
    rank_90 = int(torch.searchsorted(energy, 0.90 * energy[-1]).item()) + 1 if energy[-1] > 0 else 0
    rank_99 = int(torch.searchsorted(energy, 0.99 * energy[-1]).item()) + 1 if energy[-1] > 0 else 0
    spectrum = {'samples': X.shape[1], 'data_rank': data_rank, 'rank_90': rank_90, 'rank_99': rank_99}
    return (left, basis), means, spectrum


@torch.no_grad()
def fit_dmd_predictor(name, data):
    factors, means, spectra, validation_rows = [], [], [], []
    if name != 'vanilla_dmd':
        indices = np.flatnonzero(data['scales'] == 1) if name == 'steering_dmd_s1' else np.arange(len(data['scales']))
    for layer in tqdm(range(L), desc=f'Fitting {name}', unit='block'):
        if name == 'vanilla_dmd':
            X = torch.from_numpy(np.asarray(data['nominal'][layer])).T.to(device)
            Y = torch.from_numpy(np.asarray(data['nominal'][layer + 1])).T.to(device)
        else:
            X = torch.from_numpy(data['deviations'][layer, indices]).T.to(device)
            Y = torch.from_numpy(data['deviations'][layer + 1, indices]).T.to(device)
            applied = torch.from_numpy(data['pushes'][layer, data['cases'][indices]]).T.to(device)
            Y = Y - applied * torch.as_tensor(data['scales'][indices], dtype=Y.dtype, device=device)[None]
            keep = X.ne(0).any(dim=0)
            X, Y = X[:, keep], Y[:, keep]
        block_factors, block_means, spectrum = change_form_factors(X, Y, centered=name == 'vanilla_dmd')
        del X, Y
        factors.append(tuple(value.cpu() for value in block_factors))
        means.append(block_means)
        spectra.append({'fit': name, 'block': layer, **spectrum})
        incoming = data['validation_d'][layer].to(device)
        actual = data['validation_d'][layer + 1].to(device)
        applied = data['validation_u'][layer].to(device)
        direction = v[layer + 1]
        actual_perp = actual.double() - (actual.double() @ direction)[:, None] * direction
        denominator = actual_perp.norm(dim=-1)
        for rank in DMD_RANKS:
            predicted = apply_dmd(incoming, block_factors, rank) + applied
            error = predicted.double() - actual.double()
            error = error - (error @ direction)[:, None] * direction
            errors = error.norm(dim=-1)[denominator > 0] / denominator[denominator > 0]
            score = torch.quantile(errors, 0.5).item() if errors.numel() else np.nan
            validation_rows.append({'fit': name, 'block': layer, 'rank': rank,
                                    'actual_rank': min(rank, spectrum['data_rank']), 'err_perp': score})
    curve = pd.DataFrame(validation_rows).groupby('rank', sort=True).agg(
        err_perp=('err_perp', 'mean'), min_actual_rank=('actual_rank', 'min'),
        max_actual_rank=('actual_rank', 'max')).reset_index()
    if not np.isfinite(curve['err_perp']).any():
        raise RuntimeError(f'{name} has no finite validation score.')
    chosen_rank = int(curve.loc[curve['err_perp'].idxmin(), 'rank'])
    curve.insert(0, 'fit', name)
    curve['selected'] = curve['rank'].eq(chosen_rank)
    for spectrum in spectra:
        spectrum['selected_rank'] = min(chosen_rank, spectrum['data_rank'])
    return {'config': fit_config, 'name': name, 'chosen_rank': chosen_rank,
            'factors': [(left[:, :chosen_rank].clone(), basis[:, :chosen_rank].clone()) for left, basis in factors],
            'means': means, 'validation': curve.to_dict('records'),
            'validation_by_block': validation_rows, 'effective_ranks': spectra}


## Load or fit cached predictors

Cache hits skip both data collection and SVD fitting. Missing variants share one training-data collection; each completed fit is saved immediately. Predictions keep the two low-rank factors on the GPU and never construct dense fitted operators.


In [ ]:
dmd_caches = {}
missing = []
for name, path in cache_paths.items():
    if path.exists():
        cached = torch.load(path, map_location='cpu', weights_only=True)
        if cached['config'] != fit_config or cached['name'] != name:
            raise ValueError(f'Existing cache does not match this configuration: {path}. It will not be refitted or overwritten.')
        dmd_caches[name] = cached
        print('Loaded:', path)
    else:
        missing.append(name)
if missing:
    with TemporaryDirectory(prefix='linearity-dmd-') as directory:
        training_data = collect_dmd_data(directory, missing)
        for name in missing:
            fitted = fit_dmd_predictor(name, training_data)
            with cache_paths[name].open('xb') as stream:
                torch.save(fitted, stream)
            dmd_caches[name] = fitted
            print(f'Saved {name}: rank={fitted["chosen_rank"]}, {cache_paths[name]}')
        del training_data
dmd_factors = {name: [(left.to(device), basis.to(device)) for left, basis in cached['factors']]
               for name, cached in dmd_caches.items()}


## Validation curves and effective ranks

Each curve is the mean of per-block median one-step off-concept errors on the same validation split. The star marks the selected requested rank; the table reports the actual rank after each block's data-rank cap.


In [ ]:
validation_curve = pd.DataFrame([row for cached in dmd_caches.values() for row in cached['validation']])
effective_ranks = pd.DataFrame([row for cached in dmd_caches.values() for row in cached['effective_ranks']])
validation_curve.to_csv(DMD_DIR / f'dmd_validation_{CONCEPT}.csv', index=False)
effective_ranks.to_csv(DMD_DIR / f'dmd_effective_ranks_{CONCEPT}.csv', index=False)
print(validation_curve.round(5).to_string(index=False))
print(effective_ranks.to_string(index=False))
print(pd.DataFrame([{'fit': name, 'training_pool': len(training_pool), 'fit_prompts': len(fit_prompts),
                     'validation_prompts': len(validation_prompts), 'chosen_rank': cached['chosen_rank'],
                     'cache': str(cache_paths[name])}
                    for name, cached in dmd_caches.items()]).to_string(index=False))
fig, ax = plt.subplots(figsize=(7, 4.5))
for name in DMD_NAMES:
    curve = validation_curve.loc[validation_curve['fit'].eq(name)].sort_values('rank')
    line, = ax.plot(curve['rank'], curve['err_perp'], marker='o', label=name)
    selected = curve.loc[curve['selected']]
    ax.plot(selected['rank'], selected['err_perp'], '*', color=line.get_color(), ms=13)
ax.set_xscale('log', base=2)
ax.set_xticks(DMD_RANKS, [str(rank) for rank in DMD_RANKS])
ax.set(xlabel='Requested rank (capped per block)', ylabel='Mean block-median validation one-step error')
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
fig.savefig(DMD_DIR / f'dmd_validation_{CONCEPT}.png', dpi=150)
plt.show()


## Reference pushes, scaled replays, and predictions

Run the reference controller separately for each actuation interval. Check zero-scale equality and unit-scale replay before computing predictions. Since rollout is linear in the applied pushes, compute its unit-scale response once and multiply by each scale; one-step uses the true deviation at each scale.

The raw CSV contains all nonzero-deviation states, including the final-norm input at state index `L`. The homogeneity gap is
$$\frac{\|d_L(s)/s-d_L(1)\|}{\|d_L(1)\|}.$$
It is undefined at zero scale, recorded as NaN. Any failed sanity check stops execution before the summary and figures.

In [ ]:
metric_rows, homogeneity_rows = [], []
nonzero_scales = tuple(scale for scale in SCALES if scale != 0)
scales = torch.tensor(nonzero_scales, dtype=torch.float32, device=device)
zero_checks = replay_checks = 0
with torch.no_grad(), sdpa_kernel(SDPBackend.MATH), tqdm(
        total=len(prompts) * len(EVERY), desc='Linearity: prompt x every', unit='case') as progress:
    for prompt_id, (tokens, text) in enumerate(prompts):
        input_ids = torch.tensor([tokens], device=device)
        inputs = {'input_ids': input_ids, 'attention_mask': torch.ones_like(input_ids)}
        clean_states, layer_kwargs = forward_states(inputs, capture_kwargs=True)
        nominal = torch.stack([state[0, -1] for state in clean_states])
        nominal_norm = nominal.double().norm(dim=-1)
        prompt_text = tokenizer.decode(tokens, skip_special_tokens=False)
        for every, controller in controllers.items():
            with record_pushes(controller) as pushes, controller.active(LAMBDA):
                reference_states, _ = forward_states(inputs)
            reference = torch.stack([state[0, -1] for state in reference_states])
            del reference_states
            replays = {}
            for scale in SCALES:
                states, _ = forward_states(inputs, pushes=scale * pushes)
                replay = torch.stack([state[0, -1] for state in states])
                if scale == 0:
                    if not torch.equal(replay, nominal):
                        raise RuntimeError(f'Sanity 1 failed: prompt={prompt_id}, every={every}, zero-scale deviation.')
                    zero_checks += 1
                if scale == 1:
                    torch.testing.assert_close(replay, reference, rtol=1e-5, atol=1e-6,
                                               msg=f'Sanity 2 failed: prompt={prompt_id}, every={every}.')
                    replay_checks += 1
                replays[scale] = replay - nominal
            deviations = torch.stack([replays[scale] for scale in nonzero_scales])
            predictions = predict_deviations(clean_states, layer_kwargs, pushes, deviations, scales)
            truth = deviations.double()
            truth_norm = truth.norm(dim=-1)
            truth_perp = truth - (truth * v).sum(dim=-1, keepdim=True) * v
            truth_perp_norm = truth_perp.norm(dim=-1)
            rel_dev = truth_norm / nominal_norm
            keep = truth_norm > 0
            positions = keep.nonzero().cpu().tolist()
            for predictor, modes in predictions.items():
                for mode, predicted in modes.items():
                    predicted = predicted.double()
                    predicted_perp = predicted - (predicted * v).sum(dim=-1, keepdim=True) * v
                    predicted_perp_norm = predicted_perp.norm(dim=-1)
                    err_full = (predicted - truth).norm(dim=-1) / truth_norm
                    err_perp = torch.where(truth_perp_norm > 0,
                                           (predicted_perp - truth_perp).norm(dim=-1) / truth_perp_norm, torch.nan)
                    ratio_perp = torch.where(truth_perp_norm > 0, predicted_perp_norm / truth_perp_norm, torch.nan)
                    cosine_denominator = predicted_perp_norm * truth_perp_norm
                    cos_perp = torch.where(cosine_denominator > 0,
                                           (predicted_perp * truth_perp).sum(-1) / cosine_denominator, torch.nan).clamp(-1, 1)
                    values = torch.stack((rel_dev, err_full, err_perp, ratio_perp, cos_perp), dim=-1)[keep].cpu().numpy()
                    for (scale_index, state_index), value in zip(positions, values):
                        metric_rows.append({'prompt_id': prompt_id, 'prompt': prompt_text, 'tokens': len(tokens),
                                            'every': every, 'scale': nonzero_scales[scale_index], 'block': state_index,
                                            'predictor': predictor, 'mode': mode,
                                            **dict(zip(('rel_dev', 'err_full', 'err_perp', 'ratio_perp', 'cos_perp'), value))})
            unit_final = replays[1][-1].double()
            unit_norm = unit_final.norm()
            for scale in SCALES:
                gap = ((replays[scale][-1].double() / scale - unit_final).norm() / unit_norm).item() if scale and unit_norm > 0 else np.nan
                homogeneity_rows.append({'prompt_id': prompt_id, 'every': every, 'scale': scale,
                                         'rel_dev': (replays[scale][-1].double().norm() / nominal_norm[-1]).item(),
                                         'homogeneity_gap': gap})
            progress.set_postfix(prompt=prompt_id, every=every)
            progress.update(1)
            del predictions, replays, deviations, truth, reference, states, replay

metrics = pd.DataFrame(metric_rows)
homogeneity = pd.DataFrame(homogeneity_rows)
metrics.to_csv(RUN_DIR / f'linearity_{CONCEPT}.csv', index=False)
homogeneity.to_csv(RUN_DIR / f'linearity_{CONCEPT}_homogeneity.csv', index=False)
del metric_rows, homogeneity_rows
print(f'Sanity 1 passed: {zero_checks} zero-scale replays exactly equal nominal states.')
print(f'Sanity 2 passed: {replay_checks} unit-scale replays reproduce reference states.')


## Local-JVP sanity check

At scale 0.01, the median full-vector one-step error must be below 0.05 at every nonzero-deviation state, checked separately for each actuation interval. The full per-prompt measurements are already saved so a failure can be inspected. Do not interpret plots if this check fails.

In [ ]:
metrics = pd.read_csv(RUN_DIR / f'linearity_{CONCEPT}.csv')
small_step = metrics.loc[metrics['predictor'].eq('prompt_jacobian') & metrics['mode'].eq('one_step')
                         & metrics['scale'].eq(0.01)]
local_error = small_step.groupby(['every', 'block'])['err_full'].median()
failed = local_error.loc[~np.isfinite(local_error) | local_error.ge(0.05)]
print('Largest per-block median err_full:', local_error.groupby('every').max().to_dict())
if not failed.empty:
    print(failed.to_string())
    raise RuntimeError('Sanity 3 failed: prompt-JVP one-step error is not below 0.05 at every block. Stop here.')
print('Sanity 3 passed.')


## Final-state table

Medians over prompts at the input to the final norm. Analysis cells reload saved CSVs; after Config, run the sanity-check cell before rerunning the table and figures. Model loading and forwards are not needed for saved-data analysis.

In [ ]:
metrics = pd.read_csv(RUN_DIR / f'linearity_{CONCEPT}.csv')
homogeneity = pd.read_csv(RUN_DIR / f'linearity_{CONCEPT}_homogeneity.csv')
final = metrics.loc[metrics['block'].eq(metrics['block'].max())]
final_table = final.groupby(['every', 'predictor', 'mode', 'scale'])[['err_perp', 'cos_perp']].median()
print(final_table.round(5).to_string())
colors = dict(zip(PREDICTORS, plt.get_cmap('tab10').colors))


## Figure 1: accumulated prediction error

Final-state rollout error versus median relative deviation, with an interquartile band over prompts. The horizontal axis is logarithmic. Points remain in increasing push-scale order.

In [ ]:
fig, axes = plt.subplots(1, len(EVERY), figsize=(6 * len(EVERY), 4.5), sharey=True, squeeze=False)
for ax, every in zip(axes[0], EVERY):
    for predictor in PREDICTORS:
        cell = final.loc[final['every'].eq(every) & final['predictor'].eq(predictor) & final['mode'].eq('rollout')]
        grouped = cell.groupby('scale', sort=True)
        curve = grouped[['rel_dev', 'err_perp']].median()
        quartiles = grouped['err_perp'].quantile([0.25, 0.75]).unstack()
        ax.plot(curve['rel_dev'], curve['err_perp'], marker='o', color=colors[predictor], label=predictor)
        ax.fill_between(curve['rel_dev'], quartiles[0.25], quartiles[0.75], color=colors[predictor], alpha=0.15)
    ax.set(xscale='log', xlabel='Median relative deviation (final state)', title=f'every={every}')
    ax.grid(alpha=0.2)
axes[0, 0].set_ylabel('Rollout off-concept relative error (final state)')
axes[0, -1].legend(fontsize=8)
fig.suptitle(f'{MODEL_NAME}, {CONCEPT}: fixed-push linearity, float32')
fig.tight_layout()
fig.savefig(RUN_DIR / f'linearity_{CONCEPT}_rollout.png', dpi=150)
plt.show()


## Figure 2: isolated block error

Median one-step off-concept error at scales 0.1 and 1. Each panel fixes the actuation interval and push scale; the last state index is the final-norm input.

In [ ]:
fig, axes = plt.subplots(len(EVERY), 2, figsize=(12, 3.8 * len(EVERY)), sharex=True, sharey=True, squeeze=False)
for row, every in zip(axes, EVERY):
    for ax, scale in zip(row, (0.1, 1)):
        for predictor in PREDICTORS:
            cell = metrics.loc[metrics['every'].eq(every) & metrics['scale'].eq(scale)
                               & metrics['predictor'].eq(predictor) & metrics['mode'].eq('one_step')]
            curve = cell.groupby('block')['err_perp'].median()
            ax.plot(curve.index, curve, color=colors[predictor], label=predictor)
        ax.set(title=f'every={every}, scale={scale:g}', xlabel='State index (last = final-norm input)')
        ax.grid(alpha=0.2)
    row[0].set_ylabel('One-step off-concept relative error')
axes[0, -1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(RUN_DIR / f'linearity_{CONCEPT}_one_step.png', dpi=150)
plt.show()


## Figure 3: true-model homogeneity

Median homogeneity gap with an interquartile band over prompts. Zero scale is undefined and omitted; unit scale is identically zero by definition. This diagnostic contains no linear predictor.

The small-scale gap need not approach zero: it compares the small-push response with the finite unit-scale response, not with an infinitesimal derivative.

In [ ]:
fig, axes = plt.subplots(1, len(EVERY), figsize=(6 * len(EVERY), 4), sharey=True, squeeze=False)
for ax, every in zip(axes[0], EVERY):
    grouped = homogeneity.loc[homogeneity['every'].eq(every) & homogeneity['scale'].gt(0)].groupby('scale')
    median = grouped['homogeneity_gap'].median()
    quartiles = grouped['homogeneity_gap'].quantile([0.25, 0.75]).unstack()
    ax.plot(median.index, median, marker='o')
    ax.fill_between(median.index, quartiles[0.25], quartiles[0.75], alpha=0.15)
    ax.set(xscale='log', xlabel='Push scale', title=f'every={every}')
    ax.grid(alpha=0.2)
axes[0, 0].set_ylabel('True-model homogeneity gap (final state)')
fig.tight_layout()
fig.savefig(RUN_DIR / f'linearity_{CONCEPT}_homogeneity.png', dpi=150)
plt.show()
